<a href="https://colab.research.google.com/github/tjdux/Introduction-to-Machine-Learning-with-Python/blob/main/06_4_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8_%EC%9D%B8%ED%84%B0%ED%8E%98%EC%9D%B4%EC%8A%A4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- 파이프라인에 들어갈 추정기는 마지막 단계를 제외하고는 모두 `transform` 메서드를 가지고 있어야 함 (다음 단계를 위한 새로운 데이터 표현을 만들 수 있어야 함)
- `Pipeline.fit()`이 실행되는 동안, 각 단계에서 이전 단계의 `transform` 출력을 입력으로 받아 `fit`과 `transform` 메서드를 차례로 호출
- 마지막 단계는 `fit` 메서드만 호출

In [ ]:
def fit(self, X, y):
  X_transformed = X
  for name, estimator in self.steps[:-1]:
    X_transformed = estimator.fit_transform(X_transformed, y)
  self.steps[-1][1].fit(X_tranasformed, y) #pipeline.steps[-1][1]: 마지막 추정기
  return self

- 예측을 할때는, 마지막 단계 이전까지 `transform` 메서드를 호출한 다음 마지막 단계에서 `predict`를 호출

In [ ]:
def predict(self, X):
  X_transformed = X
  for step in self.steps[:-1]:
    X_transformed = steps[1].transform(X_transformed)
  return self.steps[-1][1].predict(X_transformed)

- 파이프라인의 마지막 단계가 `predict` 함수를 꼭 가져야 할 필요는 없지만 (스케일 변환, PCA), 파이프라인의 마지막 단계에는 최소한 `fit` 메서드가 있어야 함

## 01 make_pipeline을 사용한 파이프라인 생성
- `make_pipeline()`: 각 단계 이름에 해당 파이썬 클래스의 이름을 부여한 파이프라인을 만듦

In [3]:
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC

# 표준적인 방법
pipe_long = Pipeline([("scaler", MinMaxScaler()), ("svm", SVC(C=100))])

# 간소화된 방법
pipe_short = make_pipeline(MinMaxScaler(), SVC(C=100))

In [4]:
# make_pipeline(): 단계의 이름을 자동으로 생성
print(f"파이프라인 단계:\n{pipe_short.steps}")

파이프라인 단계:
[('minmaxscaler', MinMaxScaler()), ('svc', SVC(C=100))]


In [5]:
# 같은 파이썬 클래스를 여러 단계에서 사용하면 이름 뒤에 숫자가 추가로 붙음
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pipe = make_pipeline(StandardScaler(), PCA(n_components=2), StandardScaler())
print(f"파이프라인 단계:\n{pipe.steps}")

파이프라인 단계:
[('standardscaler-1', StandardScaler()), ('pca', PCA(n_components=2)), ('standardscaler-2', StandardScaler())]


## 02 단계 속성에 접근하기

- `named_steps`: 단계 이름을 키로 가진 딕셔너리

In [6]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()

pipe.fit(cancer.data)

# pca 단계의 두 개 주성분을 추출
components = pipe.named_steps["pca"].components_
print(f"components.shape: {components.shape}")

components.shape: (2, 30)


## 03 그리드 서치 안의 파이프라인 속성에 접근하기
- 파이프라인의 주된 목적: 그리드 서치

In [7]:
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

In [8]:
param_grid = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}

In [10]:
from sklearn.model_selection import train_test_split, GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=4
)
grid = GridSearchCV(pipe, param_grid, cv=5)
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('logisticregression',
                                        LogisticRegression(max_iter=1000))]),
             param_grid={'logisticregression__C': [0.01, 0.1, 1, 10, 100]})

In [11]:
print(f"최상의 모델:\n{grid.best_estimator_}")

최상의 모델:
Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(C=1, max_iter=1000))])


In [12]:
print(f"로지스틱 회귀 단계:\n{grid.best_estimator_.named_steps["logisticregression"]}")

로지스틱 회귀 단계:
LogisticRegression(C=1, max_iter=1000)


In [13]:
print(f"로지스틱 회귀 계수:\n{grid.best_estimator_.named_steps["logisticregression"].coef_}")

로지스틱 회귀 계수:
[[-0.4475566  -0.34609376 -0.41703843 -0.52889408 -0.15784407  0.60271339
  -0.71771325 -0.78367478  0.04847448  0.27478533 -1.29504052  0.05314385
  -0.69103766 -0.91925087 -0.14791795  0.46138699 -0.1264859  -0.10289486
   0.42812714  0.71492797 -1.08532414 -1.09273614 -0.85133685 -1.04104568
  -0.72839683  0.07656216 -0.83641023 -0.64928603 -0.6491432  -0.42968125]]
